In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
import numpy as np
from sklearn.model_selection import ParameterSampler

In [15]:
df_train = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/train_data.csv", encoding = "utf-8")
df_val = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/val_data.csv", encoding = "utf-8")
df_test = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/test_data.csv", encoding = "utf-8")

df_train['Datum'] = pd.to_datetime(df_train['Datum'], format='%Y-%m-%d')
df_val['Datum'] = pd.to_datetime(df_val['Datum'], format='%Y-%m-%d')
df_test['Datum'] = pd.to_datetime(df_test['Datum'], format='%Y-%m-%d')

In [16]:
# show all columns
pd.set_option('display.max_columns', None)
df_train.head()

,id,Datum,Warengruppe,Umsatz,KielerWoche,Bewoelkung,Temperatur,Windgeschwindigkeit,Woche,Monat,Wochentag,Feiertag,Jahreszeit,Ferien,sunny,cloudy,rainy,thunderstorm,is_weekend,sin_Monat,cos_Monat,sin_Wochentag,cos_Wochentag
0,1307011,2013-07-01,1,148.828353,0,6,17.8375,15,27,7,1,0,3,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
1,1307012,2013-07-01,2,535.856285,0,6,17.8375,15,27,7,1,0,3,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
2,1307013,2013-07-01,3,201.198426,0,6,17.8375,15,27,7,1,0,3,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
3,1307014,2013-07-01,4,65.890169,0,6,17.8375,15,27,7,1,0,3,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
4,1307015,2013-07-01,5,317.475875,0,6,17.8375,15,27,7,1,0,3,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349


In [17]:
#one hot encoding categorical variables
categorical_cols = ['Warengruppe', 'Jahreszeit']
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=False)
df_val = pd.get_dummies(df_val, columns=categorical_cols, drop_first=False)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=False)


In [18]:
#list columns
df_train.columns.tolist()

['id',
 'Datum',
 'Umsatz',
 'KielerWoche',
 'Bewoelkung',
 'Temperatur',
 'Windgeschwindigkeit',
 'Woche',
 'Monat',
 'Wochentag',
 'Feiertag',
 'Ferien',
 'sunny',
 'cloudy',
 'rainy',
 'thunderstorm',
 'is_weekend',
 'sin_Monat',
 'cos_Monat',
 'sin_Wochentag',
 'cos_Wochentag',
 'Warengruppe_1',
 'Warengruppe_2',
 'Warengruppe_3',
 'Warengruppe_4',
 'Warengruppe_5',
 'Warengruppe_6',
 'Jahreszeit_1',
 'Jahreszeit_2',
 'Jahreszeit_3',
 'Jahreszeit_4']

In [20]:
# alvo
TARGET = 'Umsatz'

# colunas que NÃO vão para o modelo
cols_drop = ['id', TARGET, 'Datum', 'Woche', 'Monat']

X_train = df_train.drop(columns=cols_drop)
y_train = df_train[TARGET]
X_val = df_val.drop(columns=cols_drop)
y_val = df_val[TARGET]

In [21]:
#check na values y_train
print(y_train.isnull().sum())

0


In [23]:
def mape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100

In [25]:
param_dist = {
    "n_estimators": [300, 600, 900, 1200],
    "max_depth": [10, 20, 30, None],
    "min_samples_leaf": [1, 2, 5, 10, 20, 50],
    "min_samples_split": [2, 5, 10, 20],
    "max_features": ["sqrt", "log2", 0.3, 0.5, 1.0],
    "bootstrap": [True, False],
}

best_mape = np.inf
best_params = None
best_model = None

for params in ParameterSampler(param_dist, n_iter=60, random_state=42):
    model = RandomForestRegressor(**params, n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    score = mape_safe(y_val, pred_val)

    if score < best_mape:
        best_mape = score
        best_params = params
        best_model = model

print("Best VAL MAPE:", best_mape)
print("Best params:", best_params)


Best VAL MAPE: 20.6566008460799
Best params: {'n_estimators': 600, 'min_samples_split': 5, 'min_samples_leaf': 20, 'max_features': 0.5, 'max_depth': 20, 'bootstrap': False}


In [26]:
y_pred_train = best_model.predict(X_train)
y_pred_val   = best_model.predict(X_val)

print('--- TREINO ---')
print(f"MAE  : {mean_absolute_error(y_train, y_pred_train):,.2f}")
print(f"R²   : {r2_score(y_train, y_pred_train):,.3f}")
print(f"MAPE : {mape_safe(y_train, y_pred_train):,.2f}%")

print('\n--- VALIDAÇÃO ---')
print(f"MAE  : {mean_absolute_error(y_val, y_pred_val):,.2f}")
print(f"R²   : {r2_score(y_val, y_pred_val):,.3f}")
print(f"MAPE : {mape_safe(y_val, y_pred_val):,.2f}%")


--- TREINO ---
MAE  : 30.51
R²   : 0.865
MAPE : 17.15%

--- VALIDAÇÃO ---
MAE  : 34.83
R²   : 0.827
MAPE : 20.66%


In [37]:
df_val_eval = df_val.copy()
df_val_eval["y_true"] = y_val.to_numpy()
df_val_eval["y_pred"] = y_pred_val

wg_cols = [c for c in df_val_eval.columns if c.startswith("Warengruppe_")]
df_val_eval["Warengruppe"] = (
    df_val_eval[wg_cols].idxmax(axis=1).str.replace("Warengruppe_", "").astype(int)
)


mape_by_group = (
    df_val_eval
    .groupby("Warengruppe")
    .apply(lambda x: mape_safe(x["y_true"], x["y_pred"]))
    .reset_index(name="MAPE")
    .sort_values("Warengruppe")
)

print(mape_by_group)
print("Macro-MAPE:", mape_by_group["MAPE"].mean())
print("Micro-MAPE:", mape_safe(df_val_eval["y_true"], df_val_eval["y_pred"]))


   Warengruppe       MAPE
0            1  19.775151
1            2  17.282304
2            3  19.851740
3            4  23.966167
4            5  18.230357
5            6  47.286790
Macro-MAPE: 24.3987515451432
Micro-MAPE: 20.6566008460799


/tmp/ipykernel_38933/2238549354.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: mape_safe(x["y_true"], x["y_pred"]))


In [38]:
# mesmas colunas que você removeu no treino
TARGET = "Umsatz"
cols_drop = ["id", TARGET, "Datum"]

df_test = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/test_data.csv", encoding="utf-8")
df_test["Datum"] = pd.to_datetime(df_test["Datum"], format="%Y-%m-%d")

X_test = df_test.drop(columns=cols_drop)

# (opcional, mas recomendado) garantir MESMA ordem de colunas do treino
X_test = X_test.reindex(columns=X_train.columns)

y_pred_test = best_model.predict(X_test)

df_test["Umsatz_Predicted"] = y_pred_test



In [39]:
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

final_rf = RandomForestRegressor(**best_params, n_jobs=-1, random_state=42)
final_rf.fit(X_trainval, y_trainval)

y_pred_test = final_rf.predict(X_test)

df_test['Umsatz_Predicted'] = y_pred_test
df_test[['id', 'Umsatz_Predicted']].to_csv(
    '/workspaces/bakery_prediction/2_BaselineModel/02_RF/predictions/04_rf_cylindrical_predictions_encoded.csv',
    index=False
)


In [40]:
wg_cols = [c for c in df_val.columns if c.startswith("Warengruppe_")]  # ou Group_
print("wg_cols:", wg_cols)

# soma por linha (deveria dar 1 em TODAS as linhas)
row_sum = df_val[wg_cols].sum(axis=1)
print(row_sum.value_counts().head(10))
print("Linhas com soma != 1:", (row_sum != 1).sum())


wg_cols: ['Warengruppe_1', 'Warengruppe_2', 'Warengruppe_3', 'Warengruppe_4', 'Warengruppe_5', 'Warengruppe_6']
1    1841
Name: count, dtype: int64
Linhas com soma != 1: 0


In [41]:
wg_in_X = [c for c in X_train.columns if c.startswith("Warengruppe_")]
print("Warengruppe no X_train:", wg_in_X)


Warengruppe no X_train: ['Warengruppe_1', 'Warengruppe_2', 'Warengruppe_3', 'Warengruppe_4', 'Warengruppe_5', 'Warengruppe_6']


In [43]:
imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.head(30))
print("\nImportância total das Warengruppe_*:", imp[imp.index.str.startswith("Warengruppe_")].sum())
print("\nImportâncias Warengruppe_*:\n", imp[imp.index.str.startswith("Warengruppe_")])


Warengruppe_2          0.451781
Warengruppe_5          0.125720
Warengruppe_4          0.092317
Jahreszeit_3           0.057376
Warengruppe_1          0.051260
Temperatur             0.037500
Wochentag              0.033907
Warengruppe_3          0.032846
is_weekend             0.025361
cos_Monat              0.022087
Ferien                 0.019382
Warengruppe_6          0.012611
sin_Monat              0.012492
cos_Wochentag          0.009965
sin_Wochentag          0.003858
Jahreszeit_2           0.003595
Bewoelkung             0.002131
Jahreszeit_1           0.001860
Windgeschwindigkeit    0.001568
Jahreszeit_4           0.001333
rainy                  0.000712
cloudy                 0.000261
KielerWoche            0.000030
sunny                  0.000027
Feiertag               0.000019
thunderstorm           0.000000
dtype: float64

Importância total das Warengruppe_*: 0.766536096935313

Importâncias Warengruppe_*:
 Warengruppe_2    0.451781
Warengruppe_5    0.125720
Warengruppe_4  